# 07 - Ablation: cost-sensitive learning vs balanced training

Compares the original training recipe against a cost-sensitive variant
(`class_weight='balanced'`). Reports predictive metrics and the deltas
on fidelity/stability/BRAS to verify that re-weighting does not degrade
explanation quality.


In [ ]:
import numpy as np
import pandas as pd

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.models import evaluate_ml, save_ml_model, train_lightgbm, train_random_forest
from xai_blockchain_framework.utils import save_csv

set_seed()
ell = np.load(PATHS.data_processed / 'elliptic_splits.npz')
eth = np.load(PATHS.data_processed / 'ethereum_splits.npz')


## Train balanced variants and evaluate

In [ ]:
rows = []
for ds, splits in [('Elliptic', ell), ('Ethereum', eth)]:
    for model_name, trainer in [
        ('RandomForest', lambda Xtr, ytr: train_random_forest(Xtr, ytr, class_weight='balanced')),
        ('LightGBM',     lambda Xtr, ytr: train_lightgbm(Xtr, ytr, class_weight='balanced')),
    ]:
        print(f'--- {model_name} balanced on {ds} ---')
        model = trainer(splits['X_train'], splits['y_train'])
        val_rep = evaluate_ml(model, splits['X_val'], splits['y_val'])
        test_rep = evaluate_ml(model, splits['X_test'], splits['y_test'], threshold=val_rep.threshold)
        for split, rep in [('Val', val_rep), ('Test', test_rep)]:
            rows.append({
                'Dataset': ds, 'Model': model_name, 'Variant': 'balanced',
                'Split': split, **rep.as_dict(),
            })
        save_ml_model(model, PATHS.models_dir / f'{ds.lower()}_{model_name.lower()}_balanced.joblib')

df = pd.DataFrame(rows)
print(df.to_string(index=False))
save_csv(df, PATHS.results_dir / 'exp_imbalance_results.csv')
